### Transform Circuits data
1. Read "bronze" races table
2. Keep ony columns for analitycs, drop column url
3. Standarize column name using snake_case (circuitId -> circuit_id, raceName -> race_name)
4. Rename column (date -> race_date)
5. Remove duplicate records
6. Transform values of columns race_name and locality to Title Case
7. Write transformed data to silver races table

##### 1. Read "bronze" circuits table

In [0]:
%run ../Workspace/common/configuration_environment

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
races_df = spark.table(bronze_table)
display(races_df)

season,round,url,raceName,date,circuitId,ingestion_timestamp,source_file
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,british grand prix,1950-05-13,silverstone,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,monaco grand prix,1950-05-21,monaco,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,indianapolis 500,1950-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,swiss grand prix,1950-06-04,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,belgian grand prix,1950-06-18,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,french grand prix,1950-07-02,reims,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,italian grand prix,1950-09-03,monza,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,swiss grand prix,1951-05-27,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,indianapolis 500,1951-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,belgian grand prix,1951-06-17,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv


##### 2. Keep ony columns for analitycs, drop column url

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(
    "season",
    "round",
    "url",
    "raceName",
    "date",
    "circuitId",
    "ingestion_timestamp", 
    "source_file"
    )

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("url"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"), 
    F.col("source_file"),
    )

##### 3. Standarize column name using snake_case (circuitId -> circuit_id)
##### 4. Rename column (lat -> latitude, long -> longitude)

In [0]:
races_renamed_df = (
    races_selected_df
    .withColumnRenamed("raceName", "race_name")
    .withColumnRenamed("circuitId", "circuit_id")
)

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({"raceName": "race_name",
                             "circuitId": "circuit_id"
                             })
)

In [0]:
display(races_renamed_df)

season,round,url,race_name,date,circuit_id,ingestion_timestamp,source_file
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,british grand prix,1950-05-13,silverstone,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,monaco grand prix,1950-05-21,monaco,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,indianapolis 500,1950-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,swiss grand prix,1950-06-04,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,belgian grand prix,1950-06-18,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,french grand prix,1950-07-02,reims,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,italian grand prix,1950-09-03,monza,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,swiss grand prix,1951-05-27,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,indianapolis 500,1951-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,belgian grand prix,1951-06-17,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv


##### 5. Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.distinct()

In [0]:
# 2nd method
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])

In [0]:
display(races_distinct_df)

season,round,url,race_name,date,circuit_id,ingestion_timestamp,source_file
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,british grand prix,1950-05-13,silverstone,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,monaco grand prix,1950-05-21,monaco,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,indianapolis 500,1950-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,swiss grand prix,1950-06-04,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,belgian grand prix,1950-06-18,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,french grand prix,1950-07-02,reims,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,italian grand prix,1950-09-03,monza,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,swiss grand prix,1951-05-27,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,indianapolis 500,1951-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,belgian grand prix,1951-06-17,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv


##### 7. Transform values of columns circuit_name and locality to Title Case

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn('race_name', F.initcap(F.col('race_name')))
)

In [0]:
display(races_final_df)

season,round,url,race_name,date,circuit_id,ingestion_timestamp,source_file
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,British Grand Prix,1950-05-13,silverstone,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,Monaco Grand Prix,1950-05-21,monaco,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,Indianapolis 500,1950-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,Swiss Grand Prix,1950-06-04,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,Belgian Grand Prix,1950-06-18,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,French Grand Prix,1950-07-02,reims,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,Italian Grand Prix,1950-09-03,monza,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,Swiss Grand Prix,1951-05-27,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,Indianapolis 500,1951-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,Belgian Grand Prix,1951-06-17,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv


##### 8. Write transformed data to silver circuits table

In [0]:
(
    races_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)    
)

In [0]:
display(
    spark.read.table(silver_table)
)

season,round,url,race_name,date,circuit_id,ingestion_timestamp,source_file
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,British Grand Prix,1950-05-13,silverstone,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,Monaco Grand Prix,1950-05-21,monaco,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,Indianapolis 500,1950-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,Swiss Grand Prix,1950-06-04,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,Belgian Grand Prix,1950-06-18,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,French Grand Prix,1950-07-02,reims,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,Italian Grand Prix,1950-09-03,monza,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,Swiss Grand Prix,1951-05-27,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,Indianapolis 500,1951-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,Belgian Grand Prix,1951-06-17,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
